# Qwen3 Embedding Spike

Goal: verify `Qwen/Qwen3-Embedding-0.6B` locally before wiring dense retrieval.

Requires `HF_TOKEN` in `.env` and the optional embedding stack installed.

In [ ]:
from pathlib import Path
import sys
from math import fsum, sqrt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from agentic_rag_knowledge_system.embeddings import Qwen3EmbeddingModel
from agentic_rag_knowledge_system.ingestion.chunking import ChunkingConfig
from agentic_rag_knowledge_system.ingestion.pipeline import ingest_text_directory
from agentic_rag_knowledge_system.settings import get_settings

settings = get_settings()
assert settings.hf_token is not None, 'Set HF_TOKEN in .env before running this notebook.'

ingestion = ingest_text_directory(
    ROOT / 'examples' / 'corpus',
    chunking_config=ChunkingConfig(chunk_size_chars=2500, chunk_overlap_chars=250),
)
len(ingestion.documents), len(ingestion.chunks)

In [ ]:
model = Qwen3EmbeddingModel(dimension=1024)
query = model.embed_query('Which strategies reduce LLM cost for repeated prompts?')
docs = model.embed_texts([chunk.text for chunk in ingestion.chunks[:5]])

len(query), [len(vector) for vector in docs]

In [ ]:
def cosine(a, b):
    numerator = fsum(x * y for x, y in zip(a, b, strict=True))
    denominator = sqrt(fsum(x * x for x in a)) * sqrt(fsum(y * y for y in b))
    return numerator / denominator

scores = [cosine(query, doc) for doc in docs]
sorted(enumerate(scores), key=lambda item: item[1], reverse=True)